# Reproducing Yuan et al. (2025) from dandiset 001882

**Paper.** Yuan M, Chatterjee S, Leys M, Odom JV, Salido EM (2025). Prevalence of IMPG1 and IMPG2
mutations leading to retinitis pigmentosa or vitelliform macular dystrophy in a cohort of patients
with inherited retinal dystrophies. *Genes* 16(1):43.
[doi:10.3390/genes16010043](https://doi.org/10.3390/genes16010043). The claims file was written
from the version of record (MDPI, open access, full text from Europe PMC, PMC11764596), with its
Tables 1 to 4 and the legends of Figures 1 to 4. The paper has no supplement and links no code.

**Dandiset.** [DANDI:001882](https://dandiarchive.org/dandiset/001882/0.260701.1757), version
0.260701.1757. The paper cites no version of the dandiset (its Data Availability Statement says no
data were shared). This is the only published version, and so the most recent; the draft holds the
same 13 assets (same asset ids and sizes), so the content the analysis uses does not differ between
them.

**What the dandiset holds.** No NWB file and no clinical recording. It holds a spreadsheet of the
seven patients (Table 1 of the paper, with an extra column of variant locations), a Word document
with tool outputs behind Tables 2 and 3 (IMPG2 scores, IMPG2 and IMPG1 stability values), the four
figures of the paper as image files (Figure 1 as a JPEG, Figures 2, 3 and 4 as TIFFs exported from
Adobe Illustrator), a script that made the folder layout, and six Windows `desktop.ini` files. The
images of Figures 2 and 3 are the composite, annotated figure panels the paper printed (one image
per modality and eye, one OCT B-scan per eye), not the clinical image files.

**Outside inputs** (the `with_inputs` claims of the claims file): the Ensembl VEP REST service
(SIFT, PolyPhen-2), the CADD API (v1.6 and v1.7), gnomAD (v3.1.2 and v4.1, GraphQL API), the ClinVar
VCF releases of November 2024, UniProt entries pinned through UniSave (Q17R60 version 129, Q9BZV3
version 146) and the InterPro API. Their versions are printed where they are read.

No random numbers are drawn anywhere in this notebook.

| Claim | Data | Status |
|---|---|---|
| cohort-prevalence | dandiset | tested (denominator from the paper) |
| all-caucasian | dandiset | blocked: data missing |
| table1-genotypes | dandiset | tested |
| table1-diagnosis-sex-age | dandiset | tested |
| inheritance-by-disease | dandiset | tested |
| table1-bcva | dandiset | tested |
| f32-sector-rp-phenotype, m83-rp-phenotype | dandiset | tested on the Figure 2 panels, by visual reading |
| m75, m82, m73, f57, m34 VMD phenotypes | dandiset | tested on the Figure 3 panels, by visual reading and FAF measures |
| rp-superior-field-no-cme | dandiset | tested on the Figure 2 panels |
| vmd-acuity-preserved | dandiset | tested |
| m73-f57-similar-lesions | dandiset | tested on the Figure 3 panels |
| m73-eog-normal | dandiset | blocked: data missing |
| other-gene-variants | dandiset | blocked: data missing |
| variant-domains | with inputs | tested (UniProt, InterPro) |
| table2-pathogenicity | with inputs | tested in part (SIFT, PolyPhen-2, CADD; MutationTaster and FATHMM not run) |
| asp423glu-novel | with inputs | tested (ClinVar, November 2024) |
| table3-destabilizing | with inputs | blocked: not runnable |
| impg1-interactions-unchanged, impg2-interactions-changed | with inputs | blocked: not runnable |
| table4-impg1-physicochemical | with inputs | tested (ProtParam definitions) |
| gnomad-frequencies | with inputs | tested (gnomAD) |

In [ ]:
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import impg

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 70)
started = time.time()
print(f"dandiset {impg.DANDISET_ID} version {impg.DANDISET_VERSION}")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
meta = impg.dandiset_metadata()
files = impg.assets()
print("Name:", meta["name"])
print("Assets summary:", {k: meta["assetsSummary"].get(k) for k in
      ("numberOfFiles", "numberOfBytes", "numberOfSubjects", "numberOfSamples", "species", "approach",
       "measurementTechnique", "dataStandard", "variableMeasured")})
print("relatedResource:", meta.get("relatedResource"), "; the paper is named only in the description:",
      "10.3390/genes16010043" in meta["description"])
print()
print(files.to_string())
print("\nNWB files:", int(files.path.str.endswith(".nwb").sum()))

In [ ]:
patients = impg.load_patients()
print("Spreadsheet shape (rows, columns), including the header row and the record-number column:", impg.table_shape())
print("Headers read (record-number column left out):", patients.attrs["header"])
print()
print(patients.drop(columns=["refraction", "clinvar", "pathogenicity"]).to_string(index=False))
print("\nEmpty columns:", [c for c in ("refraction", "clinvar", "pathogenicity") if patients[c].isna().all()])
print("Patients:", len(patients), "; by sex:", patients.sex.value_counts().to_dict(),
      "; by diagnosis:", patients.diagnosis.value_counts().to_dict())
print("Ages (from the identifier):", patients.age.tolist(), "; range", patients.age.min(), "to", patients.age.max())
print("Acuities are Snellen 20/x in every cell:",
      bool(patients.va_od.str.fullmatch(r"20/\d+").all() and patients.va_os.str.fullmatch(r"20/\d+").all()))

The spreadsheet holds the seven patients of Table 1 and nothing about the other 473 patients of the
cohort. Its first column holds what look like hospital record numbers; they are not shown here. Its
Refraction, clinVar and Pathogenicity columns are empty. Sex and age exist only as the patient label
(for example "F32"), the same label the paper uses; no visit date, birth date, race or ethnicity is
recorded. The dandiset's metadata says one subject and no species, while the files describe seven
patients.

In [ ]:
paragraphs, doc_tables, n_shapes = impg.load_docx()
print("Word document paragraphs:")
for p in paragraphs:
    print("  ", p)
print("\nEmbedded images:", n_shapes)
for i, t in enumerate(doc_tables):
    print(f"\nTable {i + 1} ({len(t)} rows):")
    for row in t:
        print("   | ".join(x.replace("\n", " / ") for x in row))

In [ ]:
headers = {p: impg.image_header(p) for p in (impg.FIG2_PATH, impg.FIG3_PATH, impg.FIG4A_PATH, impg.FIG4B_PATH)}
for p, h in headers.items():
    print(p, h)
fig1 = impg.read_image(impg.FIG1_PATH)
fig2 = impg.read_image(impg.FIG2_PATH)
fig3 = impg.read_image(impg.FIG3_PATH)
fig4a = impg.read_image(impg.FIG4A_PATH)
fig4b = impg.read_image(impg.FIG4B_PATH)
for name, a in (("Figure 1", fig1), ("Figure 2", fig2), ("Figure 3", fig3), ("Figure 4a", fig4a), ("Figure 4b", fig4b)):
    print(f"{name}: {a.shape}, {a.dtype}, values {a.min()} to {a.max()}")
digests = [impg._get(f"{impg.API}/dandisets/{impg.DANDISET_ID}/versions/{impg.DANDISET_VERSION}/assets/{a}/").json()
           ["digest"]["dandi:sha2-256"] for a in files.asset_id]
print("\nDistinct SHA-256 digests among", len(files), "assets:", len(set(digests)),
      "(the six desktop.ini files are", int(files.path.str.endswith("desktop.ini").sum()), "of them)")
print(f"loaded in {time.time() - started:.0f} s")

The image files under `sub-01/micr/` are named as BIDS microscopy samples (`sample-01` to
`sample-04b`), with `.ome.tif` in the name, but they are not OME-TIFF: they are the paper's figures
exported from Adobe Illustrator (24.2, 30 November 2024), in RGB. `sample-01` is Figure 1 (domain
maps), `sample-02` is Figure 2 (the RP patients), the file named `Fig 3. VMD` is Figure 3 (the VMD
patients), and `sample-04a` and `sample-04b` are the two halves of Figure 4 (IMPG1 and IMPG2 residue
contacts). The panels are shown below in each claim's section. The six `desktop.ini` files are
identical Google Drive folder settings. No recording is deposited twice; there is one image per
modality and eye. There is no time base in any file, so there is no clock to check.

In [ ]:
def show(fig, figure, eyes, modalities, title, step=4):
    """Panels of Figure 2 or 3 for the given eyes and modalities, downsampled by `step`."""
    f, axes = plt.subplots(len(eyes), len(modalities), figsize=(4 * len(modalities), 3.6 * len(eyes)),
                           squeeze=False)
    for i, e in enumerate(eyes):
        for j, m in enumerate(modalities):
            p = impg.panel(fig, figure, e, m)
            axes[i, j].imshow(p[::step, ::step])
            axes[i, j].set_title(f"{e} {m}  ({p.shape[1]} x {p.shape[0]} px)", fontsize=9)
            axes[i, j].axis("off")
    f.suptitle(title)
    plt.tight_layout()
    plt.show()


def readings_table(rows):
    """The author's visual readings: feature, panels looked at, what was seen, verdict."""
    df = pd.DataFrame(rows, columns=["feature", "panels", "seen", "verdict"])
    print(df.to_string(index=False))
    print("\nverdicts:", df.verdict.value_counts().to_dict())
    return df

## cohort-prevalence

> Of 480 IRD patients in the cohort, 7 (1.45%) carried IMPG1 or IMPG2 mutations; 2 (0.41%) were
> diagnosed with RP and 5 (1.04%) with VMD.

The dandiset holds only the seven carriers, so, as the criterion says, the numerators are counted
from the dataset and the denominator of 480 is taken from the paper (Methods 2.1).

In [ ]:
COHORT = 480   # Methods 2.1; not in the dandiset
carriers = patients[patients.gene.str.contains("IMPG")]
rp = carriers.diagnosis.str.contains(r"\bRP\b")
vmd = carriers.diagnosis.str.contains("VMD")
print("Carriers in the spreadsheet:", len(carriers), "; genes:", carriers.gene.value_counts().to_dict())
print("RP:", int(rp.sum()), carriers.patient[rp].tolist(), "; VMD:", int(vmd.sum()), carriers.patient[vmd].tolist())
for name, n, exp in (("IMPG carriers", len(carriers), 1.45), ("RP", int(rp.sum()), 0.41), ("VMD", int(vmd.sum()), 1.04)):
    pct = 100 * n / COHORT
    print(f"{name}: {n}/{COHORT} = {pct:.3f}% (paper {exp}%, tolerance {0.1 * exp:.3f}; "
          f"within: {abs(pct - exp) <= 0.1 * exp}; truncated to 2 decimals {np.floor(pct * 100) / 100})")
print(f"Discussion's 1 in 68: {COHORT}/{len(carriers)} = {COHORT / len(carriers):.1f}")
print("Patients in the dandiset that are not carriers:", len(patients) - len(carriers))

**Result: reproduced, with the denominator taken from the paper.** The spreadsheet holds 7
IMPG carriers, 2 with RP (F32, M83) and 5 with VMD; over the paper's cohort of 480 these are 1.458%,
0.417% and 1.042%, each within 10% of 1.45%, 0.41% and 1.04%, and 480/7 is 68.6 (the Discussion's
"1 in 68"). The dandiset holds no patient outside the seven, so the 480 cannot be checked.

## all-caucasian

> All seven patients with IMPG mutations were Caucasian.

In [ ]:
words = ("race", "ethnic", "caucas", "white", "european", "ancestry")
text = " ".join(patients.attrs["header"]).lower() + " " + meta["description"].lower()
print("Spreadsheet headers:", patients.attrs["header"])
print("Any of", words, "in the spreadsheet headers or the dandiset description:", [w for w in words if w in text])

**Result: blocked (data missing).** No file and no metadata field of the dandiset records race,
ethnicity or ancestry.

## table1-genotypes

> The seven patients carry the variants of Table 1.

Table 1 as the claims file transcribes it is the reference; the spreadsheet is compared field by
field, after normalising spaces around ";" and "&". IMPG1 is the reading of Table 1's "IMGP1" (the
claim's criterion).

In [ ]:
TABLE1 = pd.DataFrame([
    ("F32", "IMPG1", "c.2218C>T", "p.Leu740Phe", "Het; D", "Missense", "Sectoral RP", "20/25", "20/25"),
    ("M83", "IMPG2", "c.3056G>A & c.391C>T", "p.Cys1019Tyr & p.Arg131Cys", "Het for both genes; R", "Missense", "RP", "20/70", "20/30"),
    ("M75", "IMPG1", "c.1269C>A", "p.Asp423Glu", "Het; D", "Missense", "VMD", "20/40", "20/40"),
    ("M82", "IMPG2", "c.745C>T", "p.Leu249Phe", "Het; D", "Missense", "VMD", "20/30", "20/25"),
    ("M73", "IMPG2", "c.2890C>T", "p.Arg964*", "Het; D", "Nonsense", "VMD", "20/100", "20/50"),
    ("F57", "IMPG2", "c.2890C>T", "p.Arg964*", "Het; D", "Nonsense", "VMD", "20/20", "20/20"),
    ("M34", "IMPG2", "c.3262C>T", "p.Arg1088*", "Het; D", "Nonsense", "VMD", "20/20", "20/20")],
    columns=["patient", "gene", "cdna", "protein", "zygosity", "mutation_type", "diagnosis", "va_od", "va_os"])


def norm(v):
    return re.sub(r"\s*([;&])\s*", r"\1 ", str(v)).strip()


merged = TABLE1.merge(patients, on="patient", suffixes=("_paper", "_data"), how="outer", indicator=True)
print("Patients in both:", int((merged._merge == "both").sum()), "of", len(TABLE1))
geno = ["gene", "cdna", "protein", "zygosity"]
match = pd.DataFrame({c: [norm(a) == norm(b) for a, b in zip(merged[c + "_paper"], merged[c + "_data"])] for c in geno})
match.insert(0, "patient", merged.patient)
print(match.to_string(index=False))
print("Patients matching in every genotype field:", int(match[geno].all(axis=1).sum()), "of", len(match))
for c in geno:
    diff = merged[[norm(a) != norm(b) for a, b in zip(merged[c + "_paper"], merged[c + "_data"])]]
    for _, r in diff.iterrows():
        print(f"  differs: {r.patient} {c}: paper {r[c + '_paper']!r}, data {r[c + '_data']!r}")

**Result: reproduced.** All 7 patients are in the spreadsheet and match Table 1 in gene, cDNA
change, protein change and zygosity (7 of 7, no field differs; F32's gene is IMPG1 in the
spreadsheet).

## table1-diagnosis-sex-age

> F32 (female, 32 years) has sectoral RP and M83 (male, 83) RP; M75, M82, M73, F57 and M34 have VMD.

In [ ]:
d = TABLE1[["patient", "diagnosis"]].merge(
    patients[["patient", "diagnosis", "sex", "age"]].rename(columns={"sex": "sex_data", "age": "age_data"}),
    on="patient", suffixes=("_paper", "_data"))
d["sex_paper"] = d.patient.str[0]
d["age_paper"] = d.patient.str[1:].astype(int)
d["diagnosis_ok"] = d.diagnosis_paper == d.diagnosis_data
d["sex_ok"] = d.sex_paper == d.sex_data
d["age_ok"] = (d.age_data - d.age_paper).abs() <= 0.1 * d.age_paper
print(d.to_string(index=False))
print("Diagnoses matching:", int(d.diagnosis_ok.sum()), "; sexes:", int(d.sex_ok.sum()), "; ages:", int(d.age_ok.sum()), "of", len(d))
print("Columns holding a date of any kind:", [c for c in patients.attrs["header"] if "date" in c.lower()])

**Result: reproduced.** Diagnoses, sexes and ages match for 7 of 7 patients. Sex and age are
held only in the patient label, the same label the paper uses, and no date is recorded, so the two
readings of age (at the last visit, or in 2024) cannot be told apart, and the age is not an
independent measurement.

## inheritance-by-disease

> The RP mutations are all missense, dominant in IMPG1 and recessive (compound heterozygous) in
> IMPG2; all VMD mutations are dominant (heterozygous), either missense or nonsense.

In [ ]:
p = patients.assign(mode=patients.zygosity.str.extract(r";\s*([DR])\s*$")[0],
                    het=patients.zygosity.str.startswith("Het"),
                    n_variants=patients.protein.str.count("p\\."))
print(p[["patient", "gene", "protein", "zygosity", "mode", "n_variants", "mutation_type", "diagnosis"]].to_string(index=False))
isrp = p.diagnosis.str.contains(r"\bRP\b")
print("\nRP: types", p[isrp].mutation_type.unique().tolist(), "; IMPG1 modes", p[isrp & (p.gene == "IMPG1")]["mode"].tolist(),
      "; IMPG2 modes", p[isrp & (p.gene == "IMPG2")]["mode"].tolist(),
      "with", p[isrp & (p.gene == "IMPG2")].n_variants.tolist(), "variants")
print("VMD: modes", p[~isrp]["mode"].unique().tolist(), "; all heterozygous for one variant:",
      bool((p[~isrp].het & (p[~isrp].n_variants == 1)).all()), "; types", sorted(p[~isrp].mutation_type.unique()))

**Result: reproduced.** Both RP patients carry missense variants, F32 one IMPG1 variant marked
dominant and M83 two IMPG2 variants marked recessive; the 5 VMD patients each carry one heterozygous
variant marked dominant, missense or nonsense.

## table1-bcva

> BCVA (OD and OS) is F32 20/25 and 20/25; M83 20/70 and 20/30; M75 20/40 and 20/40; M82 20/30 and
> 20/25; M73 20/100 and 20/50; F57 20/20 and 20/20; M34 20/20 and 20/20.

In [ ]:
b = TABLE1[["patient", "va_od", "va_os"]].merge(patients[["patient", "va_od", "va_os"]], on="patient",
                                                suffixes=("_paper", "_data"))
b["od_ok"] = b.va_od_paper == b.va_od_data
b["os_ok"] = b.va_os_paper == b.va_os_data
print(b.to_string(index=False))
print("Eyes matching:", int(b.od_ok.sum() + b.os_ok.sum()), "of", 2 * len(b))
print("Acuities per eye in the spreadsheet: one value; visit dates recorded:", False)

**Result: reproduced.** All 14 eyes match Table 1. The spreadsheet holds one acuity per eye and
no visit date, so the readings "most recent visit" and "any visit" coincide.

## vmd-acuity-preserved

> In most VMD patients visual acuity was not significantly reduced; the only exception is M73
> (20/100 OD, 20/50 OS).

The readings named by the criterion are run: a threshold of worse than 20/40 on the better eye, and
on the worse eye thresholds of worse than 20/40, 20/50 and 20/70.

In [ ]:
v = patients[patients.diagnosis == "VMD"].copy()
v["od"] = v.va_od.map(impg.snellen_denominator)
v["os"] = v.va_os.map(impg.snellen_denominator)
v["worse"] = v[["od", "os"]].max(axis=1)
v["better"] = v[["od", "os"]].min(axis=1)
print(v[["patient", "va_od", "va_os", "worse", "better"]].to_string(index=False))
rows = []
for eye, thr in (("better", 40), ("worse", 40), ("worse", 50), ("worse", 70)):
    reduced = v.patient[v[eye] > thr].tolist()
    rows.append(dict(eye=eye, reduced_if_worse_than=f"20/{thr}", reduced=reduced,
                     n_not_reduced=len(v) - len(reduced), only_m73=reduced == ["M73"]))
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** Under all four readings M73 is the only VMD patient counted as reduced,
and the other 4 are not reduced.

## Figure 2: f32-sector-rp-phenotype, m83-rp-phenotype, rp-superior-field-no-cme

The claims are judged, as their criterion says, by looking at each modality and eye where the
Figure 2 legend marks the feature. The dandiset holds these images only as the panels of Figure 2:
for each eye one ultra-widefield colour photograph (FF), one FAF image, one Goldmann chart and one
SD-OCT B-scan with its infrared localiser. The readings below are the author's, made on these
panels at full resolution; each lists the panels it rests on. On a Goldmann chart the upper half is
the superior field.

In [ ]:
show(fig2, 2, ["F32 OD", "F32 OS"], ["FF", "FAF", "GVF", "OCT"], "Figure 2A,B: patient F32", step=3)
show(fig2, 2, ["M83 OD", "M83 OS"], ["FF", "FAF", "GVF", "OCT"], "Figure 2C,D: patient M83", step=3)

In [ ]:
for e in ("F32 OD", "F32 OS"):
    print(e, {m: impg.panel_signal(impg.panel(fig2, 2, e, m))["nonblack"].__round__(2) for m in impg.FIG2_COLS})
f32 = readings_table([
    ("inferior bone spicule pigmentation (FF)", "F32 OD FF, F32 OS FF",
     "dark spicule-shaped pigment over the lower nasal periphery, below and nasal to the disc, in both eyes", "visible"),
    ("reduced autofluorescence (FAF)", "F32 OD FAF, F32 OS FAF",
     "a dark band along the lower and nasal periphery with a sharp border, matching the pigmented area", "visible"),
    ("normal foveal architecture, no significant thinning (SD-OCT)", "F32 OD OCT, F32 OS OCT",
     "normal foveal pit, all layers and the ellipsoid zone continuous across the scan", "visible"),
    ("superior depression (GVF)", "F32 OD GVF, F32 OS GVF",
     "outer isopter reaches about 40 to 45 degrees above fixation and about 70 below, in both eyes", "visible")])

**Result: reproduced (visual reading of the Figure 2 panels).** All 4 features are visible in
both eyes: pigment in the lower nasal periphery on FF, a matching band of reduced FAF, a normal
fovea on the B-scan and a superior depression of the Goldmann field. The panels are the paper's own
figure, so this checks the text against the published images, not against independent data.

In [ ]:
m83 = readings_table([
    ("bilateral outer retinal atrophy sparing the central fovea (FF)", "M83 OD FF, M83 OS FF",
     "widespread mottled atrophy of the posterior pole and mid-periphery in both eyes", "visible"),
    ("bull's eye pattern around the fovea (FF)", "M83 OS FF (inset)",
     "a ring of atrophy around a darker centre in the inset of the left eye; not resolvable in the right eye", "visible"),
    ("360-degree bone spicules (FF)", "M83 OD FF, M83 OS FF",
     "scattered dark pigment clumps inferiorly and at the posterior pole; the superior periphery is hidden by lashes and cannot be judged",
     "not discernible"),
    ("pale optic discs (FF)", "M83 OD FF, M83 OS FF",
     "on the pseudo-colour widefield image the disc colour cannot be judged", "not discernible"),
    ("attenuated retinal vessels (FF)", "M83 OD FF, M83 OS FF",
     "vessels are thin, but no reference is available on these images to call them attenuated", "not discernible"),
    ("reduced autofluorescence (FAF)", "M83 OD FAF, M83 OS FAF",
     "patchy and confluent hypo-autofluorescence over the posterior pole and periphery in both eyes", "visible"),
    ("inferior residual crescent and small central field (GVF)", "M83 OD GVF, M83 OS GVF",
     "a central island within about 5 to 10 degrees and an inferior crescent-shaped island, in both eyes", "visible"),
    ("diffuse ellipsoid zone and outer nuclear layer loss (SD-OCT)", "M83 OD OCT, M83 OS OCT",
     "outer retina thinned with no continuous ellipsoid zone over most of both scans", "visible"),
    ("part of the central fovea spared in the left eye (SD-OCT)", "M83 OS OCT",
     "a short segment of ellipsoid zone at the arrow near the fovea of the left eye", "visible")])

**Result: partially reproduced (visual reading).** 6 of the 9 named features are visible; the
360-degree spread of bone spicules, the pale discs and the attenuated vessels cannot be judged on
the pseudo-colour widefield panels.

In [ ]:
rp_rows = readings_table([
    ("superior field limitation, F32", "F32 OD GVF, F32 OS GVF",
     "outer isopter about 40 to 45 degrees superiorly against about 70 inferiorly", "visible"),
    ("superior field limitation, M83", "M83 OD GVF, M83 OS GVF",
     "no field above the central island; the residual crescent is inferior", "visible"),
    ("no cystoid spaces in the macula, F32", "F32 OD OCT, F32 OS OCT",
     "no intraretinal hyporeflective cysts in the single B-scan of each eye", "visible"),
    ("no cystoid spaces in the macula, M83", "M83 OD OCT, M83 OS OCT",
     "no intraretinal hyporeflective cysts in the single B-scan of each eye", "visible")])
print("\nB-scans per eye available:", 1, "(the other scans of each volume are not in the dandiset)")

**Result: reproduced (visual reading).** Both RP patients have a superior field limitation on
the Goldmann charts, and none of the 4 B-scans shows cystoid spaces; only one B-scan per eye is in
the dandiset.

## Figure 3: the VMD phenotypes and m73-f57-similar-lesions

As for Figure 2, each feature is read where the Figure 3 legend's arrows point. The FAF features
are also measured: at the lesion the yellow arrow marks, the mean grey level in a disc of radius 16,
32 and 48 pixels, the 95th percentile within 16 pixels (bright foci), and the area darker than 0.85
of the surrounding fundus within 100 pixels, each against the median of an annulus 80 to 140 pixels
from the lesion (`impg.faf_measures`). The normal macula is darker than its surround on FAF, so a
ratio below 1 alone does not show a lesion; a 95th percentile above 1.1 is taken as a bright,
hyper-autofluorescent focus. These measures are a standard choice to make the reading checkable, not
the paper's method, which is clinical inspection. The empty panels are checked by their fraction of
non-black pixels.

In [ ]:
signal = pd.DataFrame([{"eye": e, **{m: round(impg.panel_signal(impg.panel(fig3, 3, e, m))["nonblack"], 3)
                                     for m in impg.FIG3_COLS}} for e in impg.FIG3_ROWS])
print("Fraction of non-black pixels per Figure 3 panel (an empty panel is near 0):")
print(signal.to_string(index=False))
faf = pd.DataFrame([{"eye": e, **impg.faf_measures(fig3, c)} for e, c in impg.FAF_LESIONS.items()]).round(3)
faf["bright_focus"] = faf.p95_r16 > 1.1
print("\nFAF measures at the arrowed lesions:")
print(faf.to_string(index=False))


def show_faf(eyes):
    f, axes = plt.subplots(1, len(eyes), figsize=(3.2 * len(eyes), 3.2))
    for ax, e in zip(axes, eyes):
        r, c = impg.FAF_LESIONS[e]
        ax.imshow(fig3[r - 150:r + 151, c - 150:c + 151])
        for rad in (16, 48, 80, 140):
            ax.add_patch(plt.Circle((150, 150), rad, fill=False, color="r" if rad < 80 else "c", lw=0.8))
        ax.set_title(e, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


show_faf(list(impg.FAF_LESIONS))

In [ ]:
show(fig3, 3, ["M75 OD", "M75 OS"], ["OCT", "FF", "FAF", "FFA"], "Figure 3A,B: patient M75")
print(faf[faf.eye.str.startswith("M75")].to_string(index=False))
m75 = readings_table([
    ("bilateral subretinal vitelliform material (SD-OCT)", "M75 OD OCT, M75 OS OCT",
     "dome-shaped subretinal material under the fovea in both eyes, larger in the left", "visible"),
    ("macular hypopigmentation (FF)", "M75 OD FF, M75 OS FF",
     "a small pale lesion at the fovea in both eyes where the arrows point", "visible"),
    ("right macula hypo-autofluorescent (FAF)", "M75 OD FAF",
     "a dark lesion with fine bright specks; mean ratio 0.75 at r16, no bright focus", "visible"),
    ("left macula hyper- (text) or hypo-autofluorescent (legend) (FAF)", "M75 OS FAF",
     "a bright central spot (95th percentile 1.32 of the surround): hyper-autofluorescent, as the text says", "visible"),
    ("early blockage and late window defects (FA)", "M75 OD FFA, M75 OS FFA",
     "one angiogram frame per eye, with a small hyperfluorescent foveal lesion in a darker macula; the early and late phases are not both shown",
     "not discernible"),
    ("superior macular nevus (infrared, FF)", "M75 OS OCT (infrared localiser), M75 OS FF",
     "a bright patch above the fovea on the left eye's infrared image and a dark grey-green lesion at the same place on its FF; none in the right eye",
     "visible")])

**Result: partially reproduced (visual reading and FAF measures).** 5 of 6 features are visible.
The left macula has a bright focus (95th percentile 1.32 of the surround), so it is
hyper-autofluorescent as the text says, not hypo-autofluorescent as the legend says; the right
macula is darker than its surround (0.75) with no bright focus. The dandiset holds one angiogram
frame per eye, so early blockage and late window defects cannot both be seen. The nevus is visible
in the left eye only.

In [ ]:
show(fig3, 3, ["M82 OD", "M82 OS"], ["OCT", "FF", "FAF", "FFA"], "Figure 3C,D: patient M82")
print("M82 FAF panels non-black fraction:", signal.set_index("eye").loc[["M82 OD", "M82 OS"], "FAF"].tolist(), "(marked NA)")
m82 = readings_table([
    ("right eye attenuated ellipsoid and retinal thinning (SD-OCT)", "M82 OD OCT",
     "thinned fovea with an irregular, faint ellipsoid zone under it", "visible"),
    ("left eye vitelliform lesion with small ellipsoid disruption (SD-OCT)", "M82 OS OCT",
     "a small hyperreflective subretinal deposit at the fovea with a gap in the ellipsoid zone", "visible"),
    ("foveal hypopigmentation in both eyes (FF)", "M82 OD FF, M82 OS FF",
     "mottled foveal pigment with red-brown and darker spots; no clear pale area", "not discernible"),
    ("bilateral window defects (FA)", "M82 OD FFA, M82 OS FFA",
     "small hyperfluorescent foveal dots in the right eye; in the left eye the arrowed area is not brighter than its surround",
     "not discernible")])

**Result: partially reproduced (visual reading).** The two SD-OCT features are visible; foveal
hypopigmentation on FF and the window defect in the left eye on FA are not discernible (2 of 4).

In [ ]:
show(fig3, 3, ["M73 OD", "M73 OS"], ["OCT", "FF", "FAF"], "Figure 3E,F: patient M73")
print(faf[faf.eye.str.startswith("M73")].to_string(index=False))
m73 = readings_table([
    ("bilateral macular subretinal hyporeflective spaces (SD-OCT)", "M73 OD OCT, M73 OS OCT",
     "an optically empty subretinal space under the fovea in both eyes", "visible"),
    ("ellipsoid disruption (SD-OCT)", "M73 OD OCT",
     "the ellipsoid zone is interrupted beside the space at the arrow in the right eye", "visible"),
    ("foveal hypopigmentation (FF)", "M73 OD FF, M73 OS FF",
     "a small pale-centred lesion at the fovea in both eyes", "visible"),
    ("scattered dot-blot hemorrhages (FF)", "M73 OD FF, M73 OS FF",
     "no red dots that can be told apart from pigment at this resolution", "not discernible"),
    ("peripheral cobblestone degeneration (FF)", "M73 OD FF, M73 OS FF",
     "many round yellow-white patches with pigmented edges in the far periphery of both eyes", "visible"),
    ("bilateral hyper-autofluorescence (FAF)", "M73 OD FAF, M73 OS FAF",
     "a bright focus in the left eye (95th percentile 1.20); in the right eye the lesion is darker than its surround with no focus above it (1.00)",
     "not discernible")])

**Result: partially reproduced (visual reading and FAF measures).** 4 of 6 features are visible.
The left macula has a bright focus (95th percentile 1.20), the right does not (1.00), so the
hyper-autofluorescence is seen in one eye, not both; dot-blot hemorrhages cannot be told apart.

In [ ]:
show(fig3, 3, ["F57 OD", "F57 OS"], ["OCT", "FF", "FAF"], "Figure 3G,H: patient F57")
fr = faf.set_index("eye")
print(faf[faf.eye.str.startswith("F57")].to_string(index=False))
print("Left minus right, mean ratio at r16, r32, r48:",
      [round(fr.loc["F57 OS", k] - fr.loc["F57 OD", k], 3) for k in ("mean_r16", "mean_r32", "mean_r48")],
      "; hypo-autofluorescent area OS / OD:", round(fr.loc["F57 OS", "hypo_area"] / fr.loc["F57 OD", "hypo_area"], 2))
f57 = readings_table([
    ("subretinal vitelliform lesions in both maculae (SD-OCT)", "F57 OD OCT, F57 OS OCT",
     "a flat subretinal hyperreflective deposit with a small cleft under the fovea in both eyes", "visible"),
    ("foveal hypopigmentation (FF)", "F57 OD FF, F57 OS FF",
     "a small darker foveal spot, not paler; no clear hypopigmentation", "not discernible"),
    ("bilateral hypo-autofluorescence (FAF)", "F57 OD FAF, F57 OS FAF",
     "a round grey foveal lesion in both eyes, darker than the surround (mean 0.82 and 0.78 at r16), no bright focus", "visible"),
    ("more pronounced in the left eye (FAF)", "F57 OD FAF, F57 OS FAF",
     "the darker area is not larger in the left eye (area ratio below 1); the eyes look alike", "not discernible")])

**Result: partially reproduced (visual reading and FAF measures).** Vitelliform lesions on SD-OCT
and hypo-autofluorescence in both eyes are visible (mean 0.82 and 0.78 of the surround at r16, no
bright focus). The left eye is darker at the centre (by 0.046 at r16) but not at r48 (0.049
brighter) and its darker area is smaller (0.77 of the right), so "more pronounced in the left eye"
is not shown; foveal hypopigmentation on FF is not discernible (2 of 4).

In [ ]:
show(fig3, 3, ["M34 OD", "M34 OS"], ["OCT", "FF", "FAF"], "Figure 3I,J: patient M34")
print(faf[faf.eye.str.startswith("M34")].to_string(index=False))
m34 = readings_table([
    ("bilateral subretinal vitelliform lesions (SD-OCT)", "M34 OD OCT, M34 OS OCT",
     "a small dome-shaped subretinal deposit at the fovea in both eyes", "visible"),
    ("glaucomatous optic nerves (FF)", "M34 OD FF, M34 OS FF",
     "the discs are small and bright on the widefield image; cupping cannot be judged", "not discernible"),
    ("mild macular hypopigmentation (FF)", "M34 OD FF, M34 OS FF",
     "a dark round foveal spot in both eyes, not a pale one", "not discernible"),
    ("faint hyper- (text) or hypo-autofluorescence (legend) (FAF)", "M34 OD FAF, M34 OS FAF",
     "a dark round foveal lesion in both eyes (mean 0.64 and 0.59 at r16, no bright focus): hypo-autofluorescent, as the legend says",
     "visible")])

**Result: partially reproduced (visual reading and FAF measures).** Bilateral subretinal lesions on
SD-OCT and FAF hypo-autofluorescence (0.64 and 0.59 of the surround, no bright focus) are visible,
the FAF matching the legend's reading, not the text's "faint hyper-autofluorescence"; glaucomatous
discs and macular hypopigmentation are not discernible (2 of 4).

In [ ]:
print(faf[faf.eye.str[:3].isin(["M73", "F57"])][["eye", "mean_r16", "p95_r16", "bright_focus", "hypo_area"]].to_string(index=False))
sim = readings_table([
    ("macular vitelliform lesions in both patients (SD-OCT)", "M73 OD/OS OCT, F57 OD/OS OCT",
     "M73: subretinal hyporeflective spaces (resorbed material); F57: subretinal hyperreflective deposits; both subfoveal", "visible"),
    ("M73 hyper-autofluorescent where F57 is hypo-autofluorescent (FAF)", "M73 OD/OS FAF, F57 OD/OS FAF",
     "M73 left eye has a bright focus and F57 none; M73 right eye has none either", "not discernible"),
    ("peripheral cobblestone degeneration in M73 and not in F57 (FF)", "M73 OD/OS FF, F57 OD/OS FF",
     "many peripheral yellow-white patches in M73; none in the F57 images (which show less periphery)", "visible")])

**Result: partially reproduced (visual reading and FAF measures).** Both patients have subfoveal
vitelliform lesions, and M73 alone has peripheral cobblestone degeneration. M73's left macula has a
bright focus (1.20) where F57 has none (0.93, 0.85), but M73's right macula has none either (1.00),
so the increased autofluorescence holds in one of his eyes.

## m73-eog-normal and other-gene-variants

Both need records the dandiset would hold as clinical reports: an electro-oculogram for M73, and
the genetic panel results beyond the IMPG variants. All text in the dandiset (the spreadsheet, the
Word document, the folder script and the dandiset metadata) is searched for them.

In [ ]:
alltext = " ".join([" ".join(map(str, impg.load_patients().drop(columns=["sex", "age"]).values.ravel())),
                    " ".join(patients.attrs["header"]), " ".join(paragraphs),
                    " ".join(" ".join(r) for t in doc_tables for r in t),
                    impg.read_bytes("setup_bids.py").decode(), str(meta)])
for term in ("EOG", "electro-oculogram", "Arden", "BEST1", "JAG1", "ALMS1", "CEP164", "COL11A1", "CPLANE1",
             "EYS", "KIF7", "NPHP4", "PHYH"):
    print(f"{term}: {len(re.findall(r'(?<![A-Za-z0-9])' + re.escape(term) + r'(?![A-Za-z0-9])', alltext, flags=re.I))} occurrences")
print("Asset types:", sorted(set(files.path.str.extract(r"(\.[^.]+)$")[0])))

**Result: blocked (data missing), both.** No EOG record and no genetic report are in the dandiset;
none of the terms searched occurs in any of its text.

## variant-domains

> The variants lie in or near named protein domains (IMPG1 p.Leu740Phe near or in the EGF-like
> domain next to SEA2, p.Asp423Glu between SEA1 and SEA2; IMPG2 p.Arg131Cys near SEA-1,
> p.Leu249Phe in SEA1, p.Arg964* in SEA2, p.Cys1019Tyr in EGF-like 1, p.Arg1088* in EGF-like 2).

Readings, from the criterion: the UniProt feature annotations of Q17R60 and Q9BZV3 (UniSave
versions current in November 2024) and the InterPro domain entries of the same sequences. A "near"
placement holds when the residue lies in no other named domain and the named domain is the closest.
The spreadsheet's own "Location of Variant" column and Figure 1 are shown beside them.

In [ ]:
seqs, feats = {}, {}
for gene in ("IMPG1", "IMPG2"):
    seqs[gene], f, release = impg.unisave(gene)
    feats[gene] = [x for x in f if x[0] in ("SIGNAL", "DOMAIN", "TRANSMEM")]
    print(gene, impg.UNISAVE[gene], release, "length", len(seqs[gene]))
    for x in feats[gene]:
        print("   UniProt", x)
ipr, ipr_rel = {}, None
for gene in ("IMPG1", "IMPG2"):
    t, ipr_rel = impg.interpro(impg.UNISAVE[gene][0])
    ipr[gene] = t[(t.db == "interpro") & (t.type == "domain")]
    print(gene, "InterPro domains:", [(r.name, r.start, r.end) for r in ipr[gene].itertuples()])
print("InterPro releases:", ipr_rel)

UNIPROT_NAMES = {"SEA 1": "SEA1", "SEA 2": "SEA2", "EGF-like 1": "EGF1", "EGF-like 2": "EGF2"}
STATED = [("IMPG1", 740, "L", "near", "EGF"), ("IMPG1", 423, "D", "between", ("SEA1", "SEA2")),
          ("IMPG2", 131, "R", "near", "SEA1"), ("IMPG2", 249, "L", "in", "SEA1"),
          ("IMPG2", 964, "R", "in", "SEA2"), ("IMPG2", 1019, "C", "in", "EGF1"),
          ("IMPG2", 1088, "R", "in", "EGF2")]


def domains(gene, reading):
    if reading == "UniProt":
        return [(UNIPROT_NAMES.get(n, n), s, e) for t, s, e, n in feats[gene] if t == "DOMAIN"]
    out, counts = [], {}
    for r in ipr[gene].sort_values("start").itertuples():
        base = "SEA" if "SEA" in r.name else "EGF"
        counts[base] = counts.get(base, 0) + 1
        out.append((f"{base}{counts[base]}", r.start, r.end))
    return out


def place(gene, pos, how, target, reading):
    doms = domains(gene, reading)
    inside = [n for n, s, e in doms if s <= pos <= e]
    dist = {n: (0 if s <= pos <= e else min(abs(pos - s), abs(pos - e))) for n, s, e in doms}
    if how == "between":
        a, b = [x for x in doms if x[0] in target], None
        ok = len(a) == 2 and a[0][2] < pos < a[1][1] and not inside
        return ok, inside, dist
    if how == "in":
        return target in inside, inside, dist
    names = [n for n in dist if n.startswith(target)] if target == "EGF" else [target]
    near = [n for n in names if n in dist]
    ok = bool(near) and not inside and min(dist[n] for n in near) == min(dist.values())
    return ok, inside, dist


rows = []
for gene, pos, aa, how, target in STATED:
    assert seqs[gene][pos - 1] == aa
    for reading in ("UniProt", "InterPro"):
        ok, inside, dist = place(gene, pos, how, target, reading)
        rows.append(dict(variant=f"{gene} {aa}{pos}", stated=f"{how} {target}", reading=reading, holds=ok,
                         inside=inside, distances=dist))
dom = pd.DataFrame(rows)
print(dom.to_string(index=False))
print("\nPlacements holding per reading:", dom.groupby("reading").holds.sum().to_dict(), "of", len(STATED))
sig2 = [x for x in feats["IMPG2"] if x[0] == "SIGNAL"][0]
print("IMPG2 R131: distance to the signal peptide", 131 - sig2[2], "and to SEA1 (UniProt)",
      dict(dom[(dom.variant == "IMPG2 R131") & (dom.reading == "UniProt")].distances.iloc[0])["SEA1"])
print("IMPG1 L740 'in' the EGF-like domain under InterPro:", "EGF1" in place("IMPG1", 740, "in", "EGF1", "InterPro")[1])
print("\nThe spreadsheet's 'Location of Variant':")
print(patients[["patient", "gene", "protein", "location"]].to_string(index=False))
plt.figure(figsize=(14, 3))
plt.imshow(fig1)
plt.axis("off")
plt.title("Figure 1 as deposited (sub-01_sample-01)")
plt.show()

**Result: partially reproduced.** 6 of 7 placements hold under each reading, never all 7. Under
UniProt, IMPG1 has no EGF-like domain, so p.Leu740Phe cannot be near one; under InterPro, IMPG1
has an EGF-like domain at 684 to 722, 18 residues before Leu740 (near, not in), but IMPG2 has only
one EGF-like domain (1010 to 1051), so p.Arg1088* is in no EGF-like 2. p.Arg131Cys is 108
residues from SEA1 and 109 from the signal peptide. The spreadsheet's own column places
p.Leu740Phe "Post EGF-like 1", after the domain.

## table2-pathogenicity

> In silico tools predict all IMPG2 variants and the novel IMPG1 p.Asp423Glu to be pathogenic, and
> IMPG1 p.Leu740Phe to be likely benign, with the scores of Table 2.

SIFT and PolyPhen-2 (HumVar, VEP's model) come from Ensembl VEP on the MANE Select transcripts
(NM_001563.4 for IMPG1, NM_016247.4 for IMPG2), on GRCh38. PolyPhen-2 HumDiv, and the outputs of
FATHMM (its score and the FATHMM-MKL and FATHMM-XF coding scores) and MutationTaster, come from
dbNSFP (through MyVariant.info), since the web tools of November 2024 cannot be called in a
pinned version; the claims file leaves open which FATHMM model and which MutationTaster output the
paper's columns are, so every output is shown. CADD comes from its versioned API, v1.6 and v1.7 on
GRCh37 and GRCh38 (GRCh37 positions from dbNSFP). The Word document's IMPG2 table, which the dandiset holds, is compared
with Table 2 as well. This cell calls VEP once per variant, which took 1 to 12 minutes while preparing.

In [ ]:
pos38 = impg.positions()
print("GRCh38 positions from dbNSFP, on the MANE Select transcripts:")
print(pos38.to_string(index=False))
t0 = time.time()
try:
    print("Ensembl release:", impg.ensembl_release())
except RuntimeError as err:
    print("Ensembl release not read:", err)
vep38 = pos38.merge(impg.vep(pos38), on="variant")
print(f"VEP GRCh38 in {time.time() - t0:.0f} s; variants answered: {int(vep38.answered.sum())} of {len(vep38)}")
print(vep38.drop(columns=["hgvs"]).to_string(index=False))
print("VEP and dbNSFP positions agree:", bool((vep38[vep38.answered].pos_vep == vep38[vep38.answered].pos).all()))
rows = []
for r in vep38.itertuples():
    ref, alt = impg.genomic_alleles(r.hgvs)
    row = dict(variant=r.variant, pos38=r.pos, ref=ref, alt=alt)
    for build, pos in (("GRCh38", r.pos), ("GRCh37", r.pos37)):
        for ver in ("v1.6", "v1.7"):
            row[f"{build} {ver}"], row[f"{build} {ver} raw"] = impg.cadd(r.chrom, pos, alt, build, ver)
    rows.append(row)
cadd = pd.DataFrame(rows)
print(cadd.to_string(index=False))

PAPER2 = pd.DataFrame([("IMPG1 p.Leu740Phe", 0.09, 0.575, 3.6, None, 22, "Likely Benign", "rs15048636"),
                       ("IMPG1 p.Asp423Glu", 0.02, 0.98, 2.5, None, 45, "Pathogenic", "Novel"),
                       ("IMPG2 p.Cys1019Tyr", 0, 0.996, 25.8, 0.0376, 46, "Pathogenic", "rs2107208112"),
                       ("IMPG2 p.Arg131Cys", 0, 0.645, 21.7, 0.0577, 91, "Pathogenic", "rs150344327"),
                       ("IMPG2 p.Leu249Phe", 0, 0.998, 22.6, 0.0076, 78, "Pathogenic", "rs376443291"),
                       ("IMPG2 p.Arg964*", None, None, 11.2, 0.011, 58, "Pathogenic", "rs267606975"),
                       ("IMPG2 p.Arg1088*", None, None, 13.4, 0.0513, 29, "Pathogenic", "rs199867882")],
                      columns=["variant", "sift", "polyphen", "cadd", "fathmm", "mutationtaster", "prediction", "id"])
cmp = PAPER2.merge(vep38[["variant", "sift", "sift_call", "polyphen", "polyphen_call", "rsid"]], on="variant",
                   suffixes=("_paper", "_vep")).merge(cadd, on="variant")


def within(paper, got):
    if paper is None or pd.isna(paper) or got is None or pd.isna(got):
        return None
    if paper == 0:
        return bool(got == 0)
    return bool(abs(got - paper) <= 0.1 * abs(paper))


cmp["sift_ok"] = [within(p, g) for p, g in zip(cmp.sift_paper, cmp.sift_vep)]
cmp["polyphen_ok"] = [within(p, g) for p, g in zip(cmp.polyphen_paper, cmp.polyphen_vep)]
for col in ("GRCh38 v1.7", "GRCh38 v1.6", "GRCh37 v1.7", "GRCh37 v1.6"):
    cmp[f"cadd_ok {col}"] = [within(p, g) for p, g in zip(cmp.cadd, cmp[col])]
cmp["id_same"] = cmp["id"] == cmp.rsid
print(cmp[["variant", "prediction", "sift_paper", "sift_vep", "sift_call", "sift_ok", "polyphen_paper", "polyphen_vep",
           "polyphen_call", "polyphen_ok", "id", "rsid", "id_same"]].to_string(index=False))
print(cmp[["variant", "cadd"] + [c for c in cmp.columns if c.startswith("cadd_ok")]].to_string(index=False))
print("\nSIFT within tolerance:", int(cmp.sift_ok.fillna(False).sum()), "of", int(cmp.sift_ok.notna().sum()),
      "; PolyPhen:", int(cmp.polyphen_ok.fillna(False).sum()), "of", int(cmp.polyphen_ok.notna().sum()))
for col in ("GRCh38 v1.7", "GRCh38 v1.6", "GRCh37 v1.7", "GRCh37 v1.6"):
    print(f"CADD PHRED within tolerance, {col}:", int(cmp[f"cadd_ok {col}"].sum()), "of 7")
print("IMPG1 SIFT with the two variants exchanged within tolerance:",
      [within(p, g) for p, g in zip(cmp.sift_paper[:2], cmp.sift_vep[:2][::-1])])
print("\ndbNSFP", impg.dbnsfp_version())
db = pd.DataFrame([{"variant": r.variant, **impg.dbnsfp(r.chrom, r.pos, *impg.genomic_alleles(r.hgvs))}
                   for r in vep38.itertuples()])
dbc = PAPER2[["variant", "polyphen", "fathmm", "mutationtaster"]].rename(
    columns={"polyphen": "polyphen_paper", "fathmm": "fathmm_paper", "mutationtaster": "mutationtaster_paper"}).merge(db, on="variant")
dbc["hdiv_ok"] = [within(p, g) if g is not None else None for p, g in zip(dbc.polyphen_paper, dbc.polyphen_hdiv)]
for col in ("fathmm", "fathmm_mkl", "fathmm_xf"):
    dbc[f"{col}_ok"] = [within(p, g) if g is not None else None for p, g in zip(dbc.fathmm_paper, dbc[col])]
dbc["mt_ok"] = [within(p, g) if g is not None else None for p, g in zip(dbc.mutationtaster_paper, dbc.mutationtaster)]
dbc["mt_x100_ok"] = [within(p, 100 * g) if g is not None else None for p, g in zip(dbc.mutationtaster_paper, dbc.mutationtaster)]
print(dbc.to_string(index=False))
print("Within tolerance: HumDiv", int(dbc.hdiv_ok.fillna(False).sum()), "of 5; FATHMM", int(dbc.fathmm_ok.fillna(False).sum()),
      ", FATHMM-MKL", int(dbc.fathmm_mkl_ok.fillna(False).sum()), ", FATHMM-XF", int(dbc.fathmm_xf_ok.fillna(False).sum()),
      "of 5; MutationTaster", int(dbc.mt_ok.fillna(False).sum()), "or x100", int(dbc.mt_x100_ok.fillna(False).sum()), "of 7")
imp1 = cmp[cmp.variant.str.startswith("IMPG1")][["variant", "prediction", "sift_call", "polyphen_call"]]
print("\nIMPG1 calls by each tool:")
print(imp1.to_string(index=False))

print("\nThe dandiset's Word document, IMPG2 table, against Table 2:")
for row in doc_tables[0][1:]:
    print("  ", " | ".join(x.replace("\n", " ") for x in row))

**Result: not reproduced.** The IMPG2 missense scores of SIFT (0, 0, 0) and PolyPhen-2 HumVar
(0.996, 0.645, 0.998) match Table 2 exactly, but the IMPG1 values do not: VEP gives SIFT 0.02 for
p.Leu740Phe and 0.09 for p.Asp423Glu, the paper's two values exchanged, PolyPhen-2 HumDiv
(dbNSFP) gives 0.575 for p.Leu740Phe as the paper does, and 3 of 5 values within tolerance, but 0.31
for p.Asp423Glu (paper 0.98), and both tools call
p.Asp423Glu tolerated or benign, against the paper's "Pathogenic". CADD PHRED matches at most 2 of
7 values under any build and version (the nonsense variants score 36 to 45 against 11.2 and 13.4),
and no FATHMM or MutationTaster output in dbNSFP matches the paper's columns. The paper's ids for
p.Leu740Phe and p.Arg964* (rs15048636, rs267606975) differ from the ids of the variants,
rs150486366 and rs267606875; the Word document gives rs267606875 for c.2890C>T.

## asp423glu-novel

> IMPG1 p.Asp423Glu, found in M75, is a novel variant with no ClinVar entry.

The variant is c.1269C>A on NM_001563.4, at GRCh38 6:76002940 G>T (VEP, above). The four ClinVar VCF
releases of November 2024 are read at that position through their tabix index.

In [ ]:
row = vep38.set_index("variant").loc["IMPG1 p.Asp423Glu"]
ref, alt = impg.genomic_alleles(row.hgvs)
print("Position", row.chrom, row.pos, ref, ">", alt, "; VEP colocated ids:", row.rsid)
recs = []
for rel in impg.CLINVAR_NOV_2024:
    recs += impg.clinvar_records(rel, row.chrom, int(row.pos))
cv = pd.DataFrame(recs)
cv["same_allele"] = (cv.ref == ref) & (cv.alt == alt)
print(cv.to_string(index=False))
print("Releases with a record of the same allele:", sorted(cv[cv.same_allele].release.unique()),
      "of", list(impg.CLINVAR_NOV_2024))
print("MD5 of the 3 November 2024 release:", impg.clinvar_md5("20241103"))

**Result: not reproduced.** IMPG1 c.1269C>A (GRCh38 6:76002940 G>T, rs758038128) is ClinVar
variation 1000102, a missense variant of uncertain significance, in all 4 ClinVar releases of
November 2024.

## table3-destabilizing, impg1-interactions-unchanged, impg2-interactions-changed

mCSM, SDM, DUET and DynaMut need the I-TASSER models of Methods 2.3, which are not deposited
anywhere; a rerun of I-TASSER (a web server with a queue of days, stochastic, with no versioned
release that can be locked) is the only reading the claims file allows. iMutant and MuPRO are web
servers too. None can be run in a locked environment here, so the three claims are blocked as not
runnable. The dandiset holds the tool outputs as tables in the Word document and the residue-contact
images of Figure 4; they are compared with Table 3 and the text below, as a check of the dandiset
against the paper, not as a test of the claims.

In [ ]:
PAPER3 = {("p.Leu740Phe", "mCSM"): -0.588, ("p.Leu740Phe", "SDM"): -0.33, ("p.Leu740Phe", "DUET"): -0.591,
          ("p.Leu740Phe", "iMutant"): -0.85, ("p.Leu740Phe", "MuPRO"): -0.930,
          ("p.Asp423Glu", "mCSM"): -0.653, ("p.Asp423Glu", "SDM"): -0.52, ("p.Asp423Glu", "DUET"): -0.138,
          ("p.Asp423Glu", "iMutant"): -0.76, ("p.Asp423Glu", "MuPRO"): -0.478,
          ("p.Cys1019Tyr", "mCSM"): -0.826, ("p.Cys1019Tyr", "SDM"): -0.78, ("p.Cys1019Tyr", "DUET"): -0.85,
          ("p.Cys1019Tyr", "iMutant"): -0.14, ("p.Cys1019Tyr", "MuPRO"): -1.266,
          ("p.Arg131Cys", "mCSM"): -0.007, ("p.Arg131Cys", "SDM"): -0.22, ("p.Arg131Cys", "DUET"): -0.018,
          ("p.Arg131Cys", "iMutant"): -1.13, ("p.Arg131Cys", "MuPRO"): -0.627,
          ("p.Leu249Phe", "mCSM"): -1.743, ("p.Leu249Phe", "SDM"): -0.66, ("p.Leu249Phe", "DUET"): -1.77,
          ("p.Leu249Phe", "iMutant"): -0.55, ("p.Leu249Phe", "MuPRO"): -1.525}
doc = {}
tools = [re.sub(r"\s*ΔΔG.*", "", h).strip() for h in doc_tables[1][0][1:]]
for row in doc_tables[1][1:]:
    for tool, cell in zip(tools, row[1:]):
        doc[(row[0], tool)] = (impg.first_number(cell), cell)
variant = None
for row in doc_tables[2][1:]:
    variant = row[0] or variant
    doc[(variant, {"MuPro": "MuPRO"}.get(row[1], row[1]))] = (impg.first_number(row[2]), row[3])
rows = [dict(variant=k[0], tool=k[1], paper=v, docx=doc.get(k, (None, ""))[0], docx_label=doc.get(k, (None, ""))[1])
        for k, v in PAPER3.items()]
t3 = pd.DataFrame(rows)
t3["agree_10pct"] = [None if d is None else abs(d - p) <= 0.1 * abs(p) for p, d in zip(t3.paper, t3.docx)]
print(t3.to_string(index=False))
print("\nTable 3 values in the Word document:", int(t3.docx.notna().sum()), "of", len(t3),
      "; agreeing within 10%:", int(t3.agree_10pct.fillna(False).sum()),
      "; positive (stabilizing) in the Word document:", t3[t3.docx > 0][["variant", "tool", "docx"]].values.tolist())
print("Table 3's own values all negative:", all(v < 0 for v in PAPER3.values()))
mean3 = t3.groupby("variant").paper.mean().sort_values()
print("Table 3 mean over the five tools (most negative first):", mean3.round(3).to_dict())

In [ ]:
f, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(fig4a)
axes[0].set_title("sub-01_sample-04a (Figure 4A,B)")
axes[1].imshow(fig4b)
axes[1].set_title("sub-01_sample-04b (Figure 4C-E)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()
print("Figure 4 files:", fig4a.shape, fig4b.shape)

**Result: blocked (not runnable), all three.** The structural models and the web tools cannot be
run here. As deposited, the Word document disagrees with Table 3 in the places printed above,
including a positive (stabilizing) SDM value for p.Asp423Glu, where Table 3 prints a negative one
and the claim says every tool gives a negative value. In the Figure 4 file of IMPG2, the images in the
top row show Arg131 and Cys131 and the middle row Cys1019 and Tyr1019, while the labels beside them
read p.Cys1019Tyr and p.Leu249Phe; the labels are one row out of step with the images (read by eye,
not measured).

## table4-impg1-physicochemical

> ProtParam gives reference IMPG1 a molecular weight of 89,387 Da, pI 4.79, instability index 53.57
> and aliphatic index 75.63; p.Asp423Glu raises the weight (89,419), pI (4.81) and instability index
> (53.76), and p.Leu740Phe raises the weight (89,421) and lowers the aliphatic index (75.14).

Readings from the criterion: (a) the canonical UniProt sequence of IMPG1 (Q17R60, UniSave version
129, current on 23 November 2024 when the paper ran ProtParam) and (b) the same without its signal
peptide (residues 1 to 20). The values follow ProtParam's definitions (`impg.protparam`). Values are
compared at the paper's printed precision (half the last digit). An Asp to Phe change at 423 is
shown as a variant, since its weight and atom count differ from Asp by the amounts Table 4 shows.

In [ ]:
s1 = seqs["IMPG1"]
sig1 = [x for x in feats["IMPG1"] if x[0] == "SIGNAL"][0]
forms = {"reference": s1, "p.Asp423Glu": impg.mutate(s1, 423, "D", "E"), "p.Leu740Phe": impg.mutate(s1, 740, "L", "F"),
         "variant p.Asp423Phe": impg.mutate(s1, 423, "D", "F")}
PAPER4 = {"molecular_weight": (89387, 89419, 89421, 0.5), "theoretical_pI": (4.79, 4.81, 4.79, 0.005),
          "atoms": (12450, 12457, 12451, 0), "instability_index": (53.57, 53.76, 53.16, 0.005),
          "aliphatic_index": (75.63, 75.63, 75.14, 0.005), "gravy": (-0.482, -0.474, -0.483, 0.0005)}
res = {}
for reading, cut in (("a full sequence", 0), ("b without signal peptide", sig1[2])):
    tab = pd.DataFrame({k: impg.protparam(v[cut:]) for k, v in forms.items()})
    res[reading] = tab
    print(f"Reading ({reading}), length {len(s1) - cut}:")
    print(tab.round(4).to_string())
rows = []
for reading, tab in res.items():
    for q, (r0, r1, r2, tol) in PAPER4.items():
        for form, exp in (("reference", r0), ("p.Asp423Glu", r1), ("p.Leu740Phe", r2)):
            got = tab.loc[q, form]
            rows.append(dict(reading=reading[0], quantity=q, form=form, paper=exp, measured=round(got, 4),
                             ok=abs(got - exp) <= tol + 1e-9))
t4 = pd.DataFrame(rows)
print(t4.pivot_table(index=["quantity", "form"], columns="reading", values="ok").to_string())
print("\nValues matching, per reading:", t4.groupby("reading").ok.sum().to_dict(), "of", len(t4) // 2)
print("Not matching under (a):", t4[(t4.reading == "a") & ~t4.ok][["quantity", "form", "paper", "measured"]].values.tolist())
a = res["a full sequence"]
print("\nDirections under (a): weight up for p.Asp423Glu", a.loc["molecular_weight", "p.Asp423Glu"] > a.loc["molecular_weight", "reference"],
      "and p.Leu740Phe", a.loc["molecular_weight", "p.Leu740Phe"] > a.loc["molecular_weight", "reference"],
      "; pI up for p.Asp423Glu", a.loc["theoretical_pI", "p.Asp423Glu"] > a.loc["theoretical_pI", "reference"],
      f"({a.loc['theoretical_pI', 'p.Asp423Glu']:.3f} against {a.loc['theoretical_pI', 'reference']:.3f})",
      "; instability up for p.Asp423Glu", a.loc["instability_index", "p.Asp423Glu"] > a.loc["instability_index", "reference"],
      "; aliphatic down for p.Leu740Phe", a.loc["aliphatic_index", "p.Leu740Phe"] < a.loc["aliphatic_index", "reference"])
print("Variant p.Asp423Phe against Table 4's p.Asp423Glu column:",
      {q: (round(a.loc[q, "variant p.Asp423Phe"], 3), PAPER4[q][1]) for q in PAPER4})

**Result: partially reproduced.** With the full UniProt sequence (reading a), 14 of 18 values
match Table 4: all 6 of the reference and all 6 of p.Leu740Phe, and for p.Asp423Glu the
instability and aliphatic indices. The p.Asp423Glu weight (89,401.4 against 89,419), pI (4.796
against 4.81), atom count (12,453 against 12,457) and GRAVY (-0.482 against -0.474) do not; an Asp
to Phe change at 423 gives Table 4's p.Asp423Glu column exactly (89,419.5, 4.81, 12,457, -0.474).
The directions the text states all hold (weight up for both, pI and instability up for
p.Asp423Glu, aliphatic index down for p.Leu740Phe). Without the signal peptide (reading b) no value
matches.

## gnomad-frequencies

> In gnomAD, IMPG2 p.Cys1019Tyr occurs in about 1 in 93,300 European (non-Finnish) individuals;
> p.Arg131Cys in about 1 in 4100 European (non-Finnish) and 1 in 3000 Middle Eastern; p.Arg964* in
> about 1 in 18,700 African/African American and 1 in 59,000 European (non-Finnish); IMPG1
> p.Leu740Phe in about 1 in 332 European (non-Finnish).

Readings from the criterion: gnomAD v3.1.2 genomes (`gnomad_r3`) and v4.1 (`gnomad_r4`: joint,
exomes, genomes), and "1 in N" as 1/allele frequency or 1/(2 x allele frequency).

In [ ]:
GN = [("IMPG2 p.Cys1019Tyr", "nfe", 93300), ("IMPG2 p.Arg131Cys", "nfe", 4100), ("IMPG2 p.Arg131Cys", "mid", 3000),
      ("IMPG2 p.Arg964*", "afr", 18700), ("IMPG2 p.Arg964*", "nfe", 59000), ("IMPG1 p.Leu740Phe", "nfe", 332)]
ids = {}
for r in vep38.itertuples():
    ref, alt = impg.genomic_alleles(r.hgvs)
    ids[r.variant] = f"{r.chrom}-{r.pos}-{ref}-{alt}"
counts = {}
for v in sorted({g[0] for g in GN}):
    for ds in ("gnomad_r3", "gnomad_r4"):
        counts[(v, ds)] = impg.gnomad(ids[v], ds)
rows = []
for v, pop, exp in GN:
    for ds, parts in (("gnomad_r3", ["genome"]), ("gnomad_r4", ["joint", "exome", "genome"])):
        for part in parts:
            ac, an = counts[(v, ds)].get(part, {}).get(pop, (0, 0))
            for reading, n in (("1/AF", an / ac if ac else np.inf), ("1/(2AF)", an / ac / 2 if ac else np.inf)):
                rows.append(dict(variant=v, pop=pop, paper=exp, dataset=ds, part=part, ac=ac, an=an, reading=reading,
                                 one_in=round(n) if np.isfinite(n) else None, ok=abs(n - exp) <= 0.1 * exp))
gn = pd.DataFrame(rows)
print("Variant ids:", ids)
print(gn.to_string(index=False))
combo = gn.groupby(["dataset", "part", "reading"]).ok.sum().sort_values(ascending=False)
print("\nValues matching per combination of readings (of 6):")
print(combo.to_string())

**Result: reproduced.** Under gnomAD v4.1 joint frequencies read as 1/(2 x allele frequency), all
6 values are within 10%: 1 in 98,335, 4,096, 3,031, 18,722, 58,997 and 332. v4.1 exomes match 4 of 6
and every other combination none; p.Cys1019Tyr is absent from v3.1.2.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| cohort-prevalence | dandiset | reproduced (denominator from the paper) |
| all-caucasian | dandiset | blocked: data missing |
| table1-genotypes | dandiset | reproduced |
| table1-diagnosis-sex-age | dandiset | reproduced |
| inheritance-by-disease | dandiset | reproduced |
| table1-bcva | dandiset | reproduced |
| f32-sector-rp-phenotype | dandiset | reproduced (visual reading of Figure 2) |
| m83-rp-phenotype | dandiset | partially reproduced |
| m75-vmd-phenotype | dandiset | partially reproduced |
| m82-vmd-phenotype | dandiset | partially reproduced |
| m73-vmd-phenotype | dandiset | partially reproduced |
| f57-vmd-phenotype | dandiset | partially reproduced |
| m34-vmd-phenotype | dandiset | partially reproduced |
| rp-superior-field-no-cme | dandiset | reproduced |
| vmd-acuity-preserved | dandiset | reproduced |
| m73-f57-similar-lesions | dandiset | partially reproduced |
| m73-eog-normal | dandiset | blocked: data missing |
| other-gene-variants | dandiset | blocked: data missing |
| variant-domains | with inputs | partially reproduced |
| table2-pathogenicity | with inputs | not reproduced |
| asp423glu-novel | with inputs | not reproduced |
| table3-destabilizing | with inputs | blocked: not runnable |
| impg1-interactions-unchanged | with inputs | blocked: not runnable |
| impg2-interactions-changed | with inputs | blocked: not runnable |
| table4-impg1-physicochemical | with inputs | partially reproduced |
| gnomad-frequencies | with inputs | reproduced |

**How the analysis differs from the paper's.** The paper graded clinical images by inspection; the
dandiset holds only the composite figure panels it published (one image per modality and eye, one
OCT B-scan per eye), so the phenotype claims were read on those panels, by the author, with FAF
lesions also measured against their surround. These claims test the text against the published
figures, not against independent recordings. The cohort denominator (480) comes from the paper.
The in silico claims were tested with pinned or versioned services (Ensembl VEP release 116,
dbNSFP 4.8a, CADD v1.6 and v1.7, gnomAD v3.1.2 and v4.1, ClinVar releases of November 2024,
UniProt entry versions of November 2024, InterPro 110.0) in place of the web tools of November
2024; MutationTaster and FATHMM are compared through dbNSFP's outputs, and the structure-based
tools (I-TASSER, mCSM, SDM, DUET, DynaMut) and iMutant and MuPRO were not run.

**What was found about the dataset.**

- It holds no recording: a spreadsheet of the seven patients, a Word document of tool outputs, and
  the paper's four figures as image files. The FF, FAF, FA, OCT and Goldmann images exist only as
  the annotated, cropped panels of Figures 2 and 3. The EOG of M73, the genetic reports (JAG1,
  BEST1 and the M83 gene list), race and ethnicity, visit dates and the other 473 patients of the
  cohort are not in it.
- The spreadsheet's first column holds what look like hospital record numbers for the seven
  patients.
- The metadata say one subject, four samples and no species; the files describe seven human
  patients. The figure files are named as BIDS microscopy samples with `.ome.tif` in the name but
  are RGB exports from Adobe Illustrator, not OME-TIFF. The paper is named in the description but
  not in `relatedResource`. Six `desktop.ini` files from Google Drive are deposited.
- The Word document differs from Table 3 in four values (SDM p.Cys1019Tyr -0.85 against -0.78,
  iMutant p.Arg131Cys -1.22 against -1.13, MuPRO p.Leu249Phe -1.2666, a copy of the p.Cys1019Tyr
  value, against -1.525, mCSM p.Asp423Glu -0.58 against -0.653) and in the sign of SDM for
  p.Asp423Glu (0.58, stabilizing, against -0.52); it has no iMutant values for IMPG1, and names
  p.Leu249Phe "p.Leu 249Tyr".
- Table 4's p.Asp423Glu column is what ProtParam gives for an Asp to Phe change; the IMPG1 SIFT
  values of Table 2 are those of the two variants exchanged; two rs ids of Table 2 are mistyped;
  and ClinVar held p.Asp423Glu before the paper called it novel.
- In the Figure 4 file of IMPG2 the variant labels are one row out of step with the images.